# genHouse3D: training on Kaggle

This notebook holds no logic: it installs the package and calls `python -m housegen.train`.

Inputs to attach (built with `make kaggle-bundle`, uploaded as private datasets):

- `genhouse3d-code`: `code.zip`, a `git archive` of the repository;
- `genhouse3d-data`: `plans.jsonl.gz`, the preprocessed plans;
- optional `genhouse3d-run`: the output of the previous session (`last.pt`, `log.csv`), to resume.

Accelerator: GPU T4 (fp16 works) or P100 (check that the preinstalled PyTorch still supports it). Sessions stop after 12 h: the run saves a checkpoint and stops after `--max-minutes`, then the next session resumes from it.

In [ ]:
%%bash
set -e
mkdir -p /kaggle/working/genHouse3D && cd /kaggle/working/genHouse3D
unzip -oq /kaggle/input/genhouse3d-code/code.zip
mkdir -p data/processed/swiss_dwellings runs/base
cp /kaggle/input/genhouse3d-data/plans.jsonl.gz data/processed/swiss_dwellings/
if [ -f /kaggle/input/genhouse3d-run/last.pt ]; then cp /kaggle/input/genhouse3d-run/last.pt /kaggle/input/genhouse3d-run/log.csv runs/base/; fi
pip install -q --no-deps -e .
pip install -q 'shapely>=2.0.6' 'trimesh>=4' networkx pyyaml
nvidia-smi --query-gpu=name,memory.total --format=csv
python -c "import torch; print(torch.__version__, torch.cuda.get_device_capability())"

In [ ]:
%%bash
cd /kaggle/working/genHouse3D
python -m housegen.train --config configs/model/base.yaml --out runs/base --max-minutes 690

In [ ]:
%%bash
# Keep only what the next session or the evaluation needs in the notebook output.
cp /kaggle/working/genHouse3D/runs/base/{last.pt,log.csv} /kaggle/working/
cp /kaggle/working/genHouse3D/runs/base/ema*.pt /kaggle/working/ 2>/dev/null || true
rm -rf /kaggle/working/genHouse3D